# Regression

## Imports

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display

## Load Datasets

In [3]:
base_dir = os.path.join(os.getcwd(), '..')
processed_data_dir = os.path.join(base_dir, 'processed_data')

open_meteo_filepath = os.path.join(processed_data_dir, 'open_meteo_processed_data.csv')
open_meteo_df = pd.read_csv(open_meteo_filepath, parse_dates=['datetime'])

ausgrid_filepath = os.path.join(processed_data_dir, 'ausgrid_processed_data.csv')
ausgrid_df = pd.read_csv(ausgrid_filepath, parse_dates=['datetime'])

### Settings

In [4]:
customer_testing_split = 0.15
datetime_testing_split = 0.15

First, we merge the smart-meter and weather data on `datetime`. A left join keeps every smart-meter row, even if there is no matching weather record. We put the merged data in time order.

In [5]:
# Merge weather and smart-meter data
data = ausgrid_df.merge(open_meteo_df, on='datetime', how='left').sort_values('datetime')

We create two separate test sets:
- Customer test: How well does the model generalize to customers it has never seen?
- Datetime test: How well does it predict a later period for customer whose earlier data is in training?

First, we choose customers to hold out. Next, we choose a later-date test period for the remaining customers. We report the actual propotions of each dataset as percentages of the original dataset. The actual proportions may differ slightly from the requested splits because customers and timestamps are kept together. Furthermore, the code chooses the latest segment by row count, not necessarily a full year, so the resulting date range may not cover all seasons.

In [6]:
rng = np.random.default_rng(42)

# Hold out whole customers, choosing the random prefix closest to the target row count.
customer_counts = data.groupby("Customer", sort=False).size()
shuffled_customers = rng.permutation(customer_counts.index.to_numpy())
customer_counts_shuffled = customer_counts.loc[shuffled_customers]
cumulative_customer_rows = customer_counts_shuffled.cumsum()

customer_target_rows = customer_testing_split * len(data)
customer_count_end = (cumulative_customer_rows - customer_target_rows).abs().argmin() + 1
customer_test_customers = shuffled_customers[:customer_count_end]

customer_test_mask = data["Customer"].isin(customer_test_customers)
customer_test_data = data.loc[customer_test_mask].copy()
remaining_data = data.loc[~customer_test_mask]

# Hold out the latest contiguous datetime segment, closest to the target row count.
rows_per_datetime = remaining_data.groupby("datetime", sort=True).size()
cumulative_tail_rows = rows_per_datetime.iloc[::-1].cumsum()

datetime_target_rows = (datetime_testing_split/(1 - customer_testing_split)) * len(remaining_data)
datetime_start = (cumulative_tail_rows.sub(datetime_target_rows).abs().idxmin())

datetime_test_mask = remaining_data["datetime"] >= datetime_start
datetime_test_data = remaining_data.loc[datetime_test_mask].copy()
train_data = remaining_data.loc[~datetime_test_mask].copy()

print(f"Train: {len(train_data) / len(data):.1%} of all rows")
print(f"Customer test: {len(customer_test_data) / len(data):.1%} of all rows")
print(f"Datetime test: {len(datetime_test_data) / len(data):.1%} of all rows")

datetime_range = datetime_test_data["datetime"].agg(["min", "max"])
print(f"Datetime test range: {datetime_range['min']} to {datetime_range['max']}")

Train: 70.0% of all rows
Customer test: 15.0% of all rows
Datetime test: 15.0% of all rows
Datetime test range: 2012-12-19 14:30:00 to 2013-06-30 23:30:00


## Average Baseline

The goal of this baseline is to predict PV generation (`Pgen`) from historical averages at the time time-of-year and time-of-day. `get_slots` converts each timestamp into a month slot (0-12) and half-hour slot (0-47).  

First we will train the model to predict power generation for unseen customers. The `monthly_profile` averages `Pgen` in the training data for each month and half-hour slot. Observations from different years, days, and customers with the same month and slot are averaged together. `Pgen` is then predicted for unseen customers (`customer_test_data`). 

In [7]:
def get_slots(data):
    months = data["datetime"].dt.month
    slots = data["datetime"].dt.hour * 2 + data["datetime"].dt.minute // 30
    return months, slots

train_months, train_slots = get_slots(train_data)
customer_test_months, customer_test_slots = get_slots(customer_test_data)

# Average historical PV generation, keeping month and half-hour slot distinct
monthly_profile = (
    pd.DataFrame({
        "month": train_months.to_numpy(),
        "slot": train_slots.to_numpy(),
        "Pgen": train_data["Pgen"].to_numpy(),
    })
    .groupby(["month", "slot"])["Pgen"]
    .mean())

test_keys = pd.MultiIndex.from_arrays([
    customer_test_months.to_numpy(),
    customer_test_slots.to_numpy(),
])

customer_test_data["Pgen_Pred_avg-baseline"] = monthly_profile.reindex(test_keys).to_numpy()

Next, we predict `Pgen` for the later-date test set, using only the training data. In this part, `monthly_profile` averages training `Pgen` separately for each customer, month, and half-hour slot. `Pgen` is then predicted for the unseen days ('datetime_test_data`).

In [8]:
datetime_test_months, datetime_test_slots = get_slots(datetime_test_data)

monthly_profile = (
    pd.DataFrame({
        "Customer": train_data["Customer"].to_numpy(),
        "month": train_months.to_numpy(),
        "slot": train_slots.to_numpy(),
        "Pgen": train_data["Pgen"].to_numpy(),
    })
    .groupby(["Customer", "month", "slot"])["Pgen"]
    .mean())

test_keys = pd.MultiIndex.from_arrays([
    datetime_test_data["Customer"].to_numpy(),
    datetime_test_months.to_numpy(),
    datetime_test_slots.to_numpy(),
])

datetime_test_data["Pgen_Pred_avg-baseline"] = monthly_profile.reindex(test_keys).to_numpy()

## Previous Day Predictor Baseline

In [14]:
temp_data = data.copy()

source = temp_data.set_index(["Customer", "datetime"])["Pgen"]
previous_keys = pd.MultiIndex.from_arrays([
    temp_data["Customer"],
    temp_data["datetime"] - pd.Timedelta(days=1)])

prediction_col = "Pgen_Pred_prev-day-baseline"
temp_data[prediction_col] = source.reindex(previous_keys).to_numpy()

customer_test_data[prediction_col] = temp_data.loc[customer_test_data.index, prediction_col]
datetime_test_data[prediction_col] = temp_data.loc[datetime_test_data.index, prediction_col]

del temp_data, source, previous_keys

## Results Dashboard

### Time Series Data Dashboard

In [ ]:
test_data_by_set = {
    "Customer test": customer_test_data,
    "Datetime test": datetime_test_data,
}

prediction_cols = sorted(
    col
    for test_data in test_data_by_set.values()
    for col in test_data.columns
    if col.startswith("Pgen_Pred_")
)
prediction_cols = sorted(set(prediction_cols))
if not prediction_cols:
    raise ValueError("No baseline prediction columns were found.")

test_set_widget = widgets.Dropdown(
    options=list(test_data_by_set),
    description="Test set:",
)

baseline_widget = widgets.Dropdown(
    options=[
        (col.removeprefix("Pgen_Pred_"), col)
        for col in prediction_cols
    ],
    value=prediction_cols[0],
    description="Baseline:",
)

initial_data = test_data_by_set[test_set_widget.value]
initial_customers = sorted(initial_data["Customer"].dropna().unique())
if not initial_customers:
    raise ValueError("No customers are available in the selected test set.")

customer_widget = widgets.Dropdown(
    options=initial_customers,
    value=initial_customers[0],
    description="Customer:",
)

initial_min = initial_data["datetime"].min().date()
initial_max = initial_data["datetime"].max().date()

start_widget = widgets.DatePicker(
    description="Start date:",
    value=initial_min,
    min=initial_min,
    max=initial_max,
)
end_widget = widgets.DatePicker(
    description="End date:",
    value=initial_max,
    min=initial_min,
    max=initial_max,
)


def update_test_set_controls(change):
    selected_data = test_data_by_set[change["new"]]
    available_customers = sorted(
        selected_data["Customer"].dropna().unique()
    )
    customer_widget.options = available_customers
    if customer_widget.value not in available_customers:
        customer_widget.value = available_customers[0]

    min_date = selected_data["datetime"].min().date()
    max_date = selected_data["datetime"].max().date()
    start_widget.min = end_widget.min = min_date
    start_widget.max = end_widget.max = max_date
    start_widget.value = min_date
    end_widget.value = max_date


test_set_widget.observe(update_test_set_controls, names="value")


def plot_timeseries(test_set, customer, baseline, start_date, end_date):
    if start_date is None or end_date is None:
        print("Select a start and end date.")
        return
    if start_date > end_date:
        print("Start date must be on or before end date.")
        return

    selected_data = test_data_by_set[test_set]
    dates = selected_data["datetime"].dt.date
    selected = selected_data.loc[
        selected_data["Customer"].eq(customer)
        & dates.between(start_date, end_date)
    ].sort_values("datetime")

    if selected.empty:
        print("No data for this customer and date range.")
        return

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(selected["datetime"], selected["Pgen"], label="Actual Pgen")
    ax.plot(
        selected["datetime"],
        selected[baseline],
        label=f"Predicted Pgen ({baseline.removeprefix('Pgen_Pred_')})",
    )
    ax.set(
        title=f"{test_set} — Customer {customer}",
        xlabel="Datetime",
        ylabel="Pgen",
    )
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.show()


controls = widgets.VBox([
    widgets.HBox([test_set_widget, customer_widget, baseline_widget]),
    widgets.HBox([start_widget, end_widget]),
])

output = widgets.interactive_output(
    plot_timeseries,
    {
        "test_set": test_set_widget,
        "customer": customer_widget,
        "baseline": baseline_widget,
        "start_date": start_widget,
        "end_date": end_widget,
    },
)

display(controls, output)

Output()

### Scatterplot Dashboard

In [28]:
def plot_scatter(test_set, customer, baseline, start_date, end_date, max_points):
    if start_date is None or end_date is None:
        print("Select a start and end date.")
        return
    if start_date > end_date:
        print("Start date must be on or before end date.")
        return

    selected_data = test_data_by_set[test_set]
    dates = selected_data["datetime"].dt.date
    mask = dates.between(start_date, end_date)

    if customer != "All customers":
        mask &= selected_data["Customer"].eq(customer)

    selected = selected_data.loc[mask, ["Pgen", baseline]].dropna()

    if selected.empty:
        print("No valid actual-predicted pairs for this selection and date range.")
        return

    if len(selected) > max_points:
        selected = selected.sample(n=max_points, random_state=42)

    actual = selected["Pgen"].to_numpy()
    predicted = selected[baseline].to_numpy()

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(
        actual,
        predicted,
        alpha=0.25 if customer == "All customers" else 0.35,
        s=4 if customer == "All customers" else 12,
        rasterized=True,
        label=f"Observations (n={len(selected):,})",
    )

    if len(selected) >= 2 and np.ptp(actual) > 0:
        slope, intercept = np.polyfit(actual, predicted, 1)
        fitted = slope * actual + intercept
        ss_res = np.sum((predicted - fitted) ** 2)
        ss_tot = np.sum((predicted - predicted.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

        x_line = np.linspace(actual.min(), actual.max(), 100)
        ax.plot(
            x_line,
            slope * x_line + intercept,
            color="red",
            label=f"Best fit: y = {slope:.3f}x {intercept:+.3f}\n$R^2$ = {r2:.3f}",
        )

    limits = [
        min(actual.min(), predicted.min()),
        max(actual.max(), predicted.max()),
    ]
    ax.plot(limits, limits, "--", color="gray", alpha=0.7, label="Perfect prediction")
    customer_label = customer if customer == "All customers" else f"Customer {customer}"
    ax.set(
        title=f"{test_set} — {customer_label}",
        xlabel="Actual Pgen",
        ylabel=f"Predicted Pgen ({baseline.removeprefix('Pgen_Pred_')})",
    )
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()

In [ ]:
from IPython.display import clear_output

def available_scatter_points(test_set, baseline):
    return int(
        test_data_by_set[test_set][["Pgen", baseline]]
        .dropna()
        .shape[0]
    )


initial_test_set = test_set_widget.value
initial_baseline = baseline_widget.value
initial_customers = sorted(
    test_data_by_set[initial_test_set]["Customer"].dropna().unique()
)
initial_max_points = max(
    1, available_scatter_points(initial_test_set, initial_baseline)
)

scatter_customer_widget = widgets.Dropdown(
    options=["All customers", *initial_customers],
    value="All customers",
    description="Customer:",
)

scatter_points_widget = widgets.IntSlider(
    value=min(20_000, initial_max_points),
    min=1,
    max=initial_max_points,
    step=max(1, initial_max_points // 100),
    description="Max points:",
    continuous_update=False,
)


def update_scatter_controls(change):
    selected_set = test_set_widget.value
    customers = sorted(
        test_data_by_set[selected_set]["Customer"].dropna().unique()
    )
    scatter_customer_widget.options = ["All customers", *customers]
    if scatter_customer_widget.value not in scatter_customer_widget.options:
        scatter_customer_widget.value = "All customers"

    max_points = max(
        1, available_scatter_points(selected_set, baseline_widget.value)
    )
    scatter_points_widget.max = max_points
    scatter_points_widget.step = max(1, max_points // 100)
    scatter_points_widget.value = min(scatter_points_widget.value, max_points)


test_set_widget.observe(update_scatter_controls, names="value")
baseline_widget.observe(update_scatter_controls, names="value")


plot_button = widgets.Button(
    description="Plot",
    button_style="primary",
    icon="bar-chart",
)

scatter_output = widgets.Output()


def on_plot_clicked(_):
    plot_button.disabled = True
    plot_button.description = "Generating Plot"

    try:
        with scatter_output:
            clear_output(wait=True)
            plot_scatter(
                test_set=test_set_widget.value,
                customer=scatter_customer_widget.value,
                baseline=baseline_widget.value,
                start_date=start_widget.value,
                end_date=end_widget.value,
                max_points=scatter_points_widget.value,
            )
    finally:
        plot_button.disabled = False
        plot_button.description = "Plot"


plot_button.on_click(on_plot_clicked)

scatter_controls = widgets.VBox([
    widgets.HBox([test_set_widget, scatter_customer_widget, baseline_widget]),
    widgets.HBox([start_widget, end_widget, scatter_points_widget]),
    plot_button,
])

display(scatter_controls, scatter_output)

Output()